
# 06 — Cross-City and Cross-Recording Validation

This notebook explains how the final four-profile model is validated when a
complete city or recording is excluded from training.

The agreement target is the Stage 5 reference partition. It is not a
ground-truth supervised label.


In [1]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES = PROJECT_ROOT / "outputs" / "tables"

city_results = pd.read_csv(TABLES / "stage6_leave_one_city_out.csv")
recording_results = pd.read_csv(
    TABLES / "stage6_leave_one_recording_out.csv"
)
summary = pd.read_csv(TABLES / "stage6_validation_summary.csv")
decision = pd.read_csv(TABLES / "stage6_validation_decision.csv")

print("City splits:", len(city_results))
print("Recording splits:", len(recording_results))
decision


City splits: 4
Recording splits: 56


,selected_model,preprocessing,validation_decision,city_gate_passed,recording_gate_passed,interpretation
0,KMeans k=4,1%-99% winsorization + RobustScaler + PCA,Validated with context sensitivity,True,True,The four-profile solution is strongly reproduc...



## 1. End-to-end train-only design

For every validation split, winsorization bounds, RobustScaler, PCA, and
K-Means are fitted using training rows only. The held-out context is only
transformed and predicted.


In [2]:

city_results[
    [
        "held_out_city",
        "n_test",
        "pca_components",
        "pca_retained_variance",
        "ari",
        "aligned_accuracy",
        "balanced_accuracy",
        "semantic_mae_mean",
        "semantic_profile_correlation_mean",
    ]
]


,held_out_city,n_test,pca_components,pca_retained_variance,ari,aligned_accuracy,balanced_accuracy,semantic_mae_mean,semantic_profile_correlation_mean
0,Changchun,8665,5,0.911610,0.685803,0.835199,0.850012,0.166599,0.849935
1,Chongqing,1928,5,0.916534,0.965776,0.984959,0.956811,0.216744,0.885882
2,Tianjin,4533,5,0.913677,0.885160,0.956541,0.917634,0.236238,0.844910
3,Xi'an,4822,5,0.929855,0.876594,0.951265,0.940420,0.113016,0.913717



## 2. Leave-one-recording-out distribution

A high median shows that the model is usually reproduced when an entire
recording is missing. The minimum identifies an honest context-sensitive case.


In [3]:

recording_results[
    ["ari", "aligned_accuracy", "balanced_accuracy"]
].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])


,ari,aligned_accuracy,balanced_accuracy
count,56.000000,56.000000,56.000000
mean,0.968895,0.988377,0.980685
std,0.053026,0.021888,0.035892
min,0.608220,0.836192,0.759143
10%,0.948547,0.982203,0.966756
25%,0.964879,0.988176,0.979059
50%,0.977425,0.991604,0.990009
75%,0.988599,0.995321,0.995299
90%,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000


In [4]:

recording_results.nsmallest(
    10,
    "ari",
)[
    [
        "city",
        "recording_id",
        "n_test",
        "ari",
        "aligned_accuracy",
        "balanced_accuracy",
    ]
]


,city,recording_id,n_test,ari,aligned_accuracy,balanced_accuracy
0,Changchun,changchun_pudong_507_009,1166,0.608220,0.836192,0.759143
17,Chongqing,6_29_NR_4,161,0.892942,0.962733,0.888889
1,Changchun,changchun_pudong_507_010,1207,0.940052,0.975973,0.976271
31,Tianjin,8_6_2,209,0.940186,0.980861,0.960164
37,Tianjin,8_9_1,208,0.940536,0.975962,0.973039
26,Tianjin,8_3_4,241,0.947653,0.983402,0.977944
54,Xi'an,xian_415_n2,416,0.949442,0.983173,0.978736
41,Xi'an,xian_412_m1,350,0.950391,0.982857,0.984375
25,Tianjin,8_3_3,246,0.950879,0.983740,0.973045
3,Changchun,changchun_pudong_523_005,1084,0.953750,0.981550,0.982928



## 3. Important interpretation

ARI and aligned accuracy measure reproducibility against the Stage 5
partition. They do not prove that the profiles are ground-truth classes.

Semantic profile metrics compare held-out profile medians with the global
profile medians in original behavioral units.


In [5]:

summary


,validation_level,splits,ari_mean,ari_median,ari_min,aligned_accuracy_mean,aligned_accuracy_min,balanced_accuracy_mean,semantic_mae_mean,semantic_profile_correlation_mean,splits_recovering_all_reference_profiles,ari_10th_percentile,aligned_accuracy_median,recordings_with_ari_below_0_60,recordings_with_ari_at_least_0_80
0,leave_one_city_out,4,0.853333,0.880877,0.685803,0.931991,0.835199,0.916219,0.183149,0.873611,4.0,NaN,NaN,NaN,NaN
1,leave_one_recording_out,56,0.968895,0.977425,0.608220,0.988377,NaN,0.980685,0.288822,0.791743,NaN,0.948547,0.991604,0.0,55.0



## Conclusion

**Stage 6 decision: Validated with context sensitivity**

The four-profile solution is recording-robust and transferable across cities,
but Changchun and one Changchun recording show stronger context sensitivity.
The final report must present both stability and this limitation.
